# Ch.10 — Production ML Monitoring & A/B Testing (Azure Supplement)

**Azure-specific extension:** This notebook adapts the main `notebook.ipynb` concepts to Azure ML infrastructure. Instead of local MLflow model serving and Evidently monitoring, we explore:

- **Azure ML Online Endpoints** for model deployment (v1 + v2)
- **Azure ML Traffic Splitting** for A/B testing (10% → 50% → 100% gradual rollout)
- **Azure Application Insights** for monitoring (request rate, latency, errors, custom metrics)
- **Azure Monitor Alerts** for automated detection (accuracy drops, latency spikes)
- **Automated Rollback** with Azure ML deployment config

**Prerequisites:**
- Complete the main `notebook.ipynb` first (understands Evidently, A/B testing, drift detection)
- Azure subscription (Free Tier sufficient for demo — uses online endpoints)
- Azure ML SDK installed: `pip install azure-ai-ml azure-identity`

**What you'll build:**
1. Deploy BERT sentiment classifier to Azure ML online endpoints (v1 + v2)
2. Configure A/B traffic splitting (gradual rollout from 10% to 100%)
3. Monitor with Application Insights (infrastructure + custom metrics)
4. Create Azure Monitor alerts (trigger on accuracy drop)
5. Implement automated rollback script (revert to v1 on alert)
6. Track inference costs (v1 vs. v2 cost comparison)

**Running example:** Same BERT sentiment classifier from main notebook, deployed to Azure ML.

**Cost estimate (for reference):**
- Azure ML online endpoint: $0.10/hr per instance (Standard_DS2_v2)
- Application Insights: First 5GB telemetry free/month, then $2.30/GB
- Total for 2-week A/B test: ~$70 (3 instances × 24hr × 14 days × $0.10/hr)

In [ ]:
# TODO: Implement this cell


In [ ]:
# TODO: Implement this cell


## 1 · Deploy Model to Azure ML Online Endpoints (v1 + v2)

**Azure ML Online Endpoints** = Managed, scalable inference service (replaces local MLflow serving)

**Key features:**
- **Managed infrastructure:** No need to provision VMs or Kubernetes clusters
- **Auto-scaling:** Scale from 1 to 100+ instances based on traffic
- **Multiple deployments:** Run v1 and v2 side-by-side on same endpoint (A/B testing)
- **Blue-green deployment:** Zero-downtime updates (traffic routing)

**Running example:** Deploy BERT sentiment classifier (v1 + v2) to Azure ML.

**Deployment strategy:**
1. **Create endpoint** (unique URL: `https://<endpoint-name>.<region>.inference.ml.azure.com`)
2. **Deploy v1** (baseline model from Ch.9 model registry)
3. **Deploy v2** (improved model from hyperparameter sweep)
4. **Configure traffic splitting:** v1=90%, v2=10% (A/B test begins)

**Cost note:** Azure ML online endpoints charge per instance-hour (e.g., $0.10/hr for Standard_DS2_v2).

In [ ]:
# TODO: Implement this cell


## 2 · Azure ML Traffic Splitting (A/B Test Configuration)

**Traffic splitting** = Route % of requests to each deployment (A/B testing, canary rollout)

**Gradual rollout strategy:**
1. **Phase 1 (Day 1-3):** v1=90%, v2=10% — Monitor for regressions
2. **Phase 2 (Day 4-7):** v1=50%, v2=50% — If metrics look good, increase v2 traffic
3. **Phase 3 (Day 8+):** v1=0%, v2=100% — Full rollout, retire v1

**Rollback strategy:** If v2 shows issues, revert to v1=100% immediately.

**Running example:** Simulate gradual rollout with traffic updates every 3 days.

In [ ]:
# TODO: Implement this cell


## 3 · Monitor with Azure Application Insights

**Application Insights** = Azure's monitoring service (infrastructure + custom metrics in one place)

**Out-of-box metrics (no code required):**
- **Request rate:** Requests/second to endpoint
- **Latency:** Average response time (p50, p95, p99)
- **Error rate:** HTTP 4xx/5xx responses
- **Availability:** Uptime % (SLA tracking)

**Custom metrics (requires instrumentation):**
- Model accuracy, F1 score (per deployment)
- Feature drift, prediction drift
- Business metrics (user satisfaction, conversions)

**Running example:** Query Application Insights for v1 vs. v2 latency comparison.

In [ ]:
# TODO: Implement this cell


## 4 · Custom Metrics (Accuracy, F1) Logged to Application Insights

**Custom metrics** = Model-specific performance metrics (not available in infrastructure monitoring)

**Implementation approaches:**
1. **Instrumentation in scoring script:** Log metrics during inference (adds latency)
2. **Async logging:** Log predictions, compute metrics offline (no latency penalty)
3. **Batch evaluation:** Sample 1% of requests, compute metrics every hour

**Running example:** Log accuracy + F1 to Application Insights (async approach).

**Note:** Ground truth required for accuracy calculation (use labeled feedback or human review).

In [ ]:
# TODO: Implement this cell


## 5 · Azure Monitor Alerts (Trigger on Accuracy Drop)

**Azure Monitor Alerts** = Automated detection + notification when metrics cross thresholds

**Alert configuration:**
- **Condition:** `custom metric 'model_accuracy' < 85%` (averaged over 10 minutes)
- **Evaluation frequency:** Every 5 minutes
- **Action:** Send email/SMS/webhook (e.g., trigger automated rollback)
- **Severity:** Critical (1) — requires immediate response

**Running example:** Create alert rule for v2 accuracy drop below 85%.

In [ ]:
# TODO: Implement this cell


## 6 · Automated Rollback with Azure ML Deployment Config

**Automated rollback** = Revert to previous deployment when alert fires

**Trigger options:**
1. **Manual:** Engineer triggers rollback via CLI/Portal
2. **Semi-automated:** Alert fires → engineer approves → rollback executes
3. **Fully automated:** Alert fires → webhook calls Azure Function → rollback executes

**Rollback strategies:**
- **Instant cutover:** v2=10% → v1=100% (safest, no downtime)
- **Gradual revert:** v2=10% → v2=0% over 10 minutes
- **Blue-green:** Keep v1 running at 0% during A/B test (instant switch back)

**Running example:** Automated rollback script triggered by Azure Monitor webhook.

In [ ]:
# TODO: Implement this cell


## 7 · Cost Comparison (v1 vs. v2 Inference Costs)

**Production ML cost tracking** = Monitor inference costs per model version

**Cost factors:**
- **Compute:** Instance type × instance count × uptime
- **Storage:** Model artifacts, logs, metrics (Azure Blob Storage)
- **Networking:** Data transfer (egress), Application Insights telemetry
- **API calls:** Azure ML online endpoint requests (free tier: 15 compute hours/month)

**Cost optimization strategies:**
- **Auto-scaling:** Scale instances based on traffic (min=1, max=10)
- **Spot instances:** Use Azure Spot VMs for non-prod (70% discount)
- **Model optimization:** Quantization, pruning, distillation (reduce compute)
- **Traffic shaping:** Route simple requests to cheap model, complex to expensive

**Running example:** Compare v1 vs. v2 daily inference costs during A/B test.

In [ ]:
# TODO: Implement this cell


## Summary & Next Steps

**What we built:**
1. ✅ Deployed BERT sentiment classifier to Azure ML online endpoints (v1 + v2)
2. ✅ Configured A/B traffic splitting (10% → 50% → 100% gradual rollout)
3. ✅ Monitored production metrics with Application Insights (latency, error rate)
4. ✅ Logged custom metrics (accuracy, F1) to Application Insights
5. ✅ Created Azure Monitor alerts (trigger on accuracy drop)
6. ✅ Implemented automated rollback script (revert to v1 on alert)
7. ✅ Tracked inference costs (compute, storage, telemetry)

**Key differences from local MLflow monitoring:**
- **Managed infrastructure:** No need to run Prometheus/Grafana/Evidently servers
- **Integrated monitoring:** Application Insights = infrastructure + custom metrics in one place
- **Cloud-scale traffic splitting:** Handle millions of requests with auto-scaling
- **Cost visibility:** Track spend per model version (optimize ROI)

**Production checklist:**
- [ ] Configure action group for alerts (email, SMS, webhook)
- [ ] Set up Azure Function for automated rollback (webhook → function → rollback)
- [ ] Enable Azure ML model monitoring (data drift, prediction drift)
- [ ] Configure auto-scaling (min=1, max=10 instances)
- [ ] Set cost budget alerts (Azure Cost Management)
- [ ] Document rollback runbook (manual steps if automation fails)

**Next steps:**
1. **Scale to multi-model serving:** Deploy multiple models to one endpoint (model routing)
2. **Advanced A/B testing:** Multi-armed bandit (Thompson sampling) for traffic allocation
3. **Feature store integration:** Version input features alongside model versions
4. **Continuous retraining:** Automated retraining pipeline (detect drift → retrain → deploy)

**Cost savings tips:**
- Use **Spot instances** for non-prod endpoints (70% discount)
- Enable **auto-scaling** to scale to zero during low traffic
- **Quantize models** to INT8 (2× faster, 75% smaller, <1% accuracy loss)
- Use **batch inference** for non-real-time workloads (10× cheaper than online endpoints)

🎉 **Congratulations!** You've deployed a production ML monitoring system with automated rollback on Azure ML.